In [1]:
# Pregunta analítica: ¿qué recomendación recurrente debe emitirse para cada solicitud de crédito, cuándo debe escalarse y quién conserva la autoridad de decisión?

import json
from pathlib import Path
import sys
import pandas as pd
root = Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
from main import apply_review_policy, policy_contract


In [2]:
# El puntaje predictivo es evidencia; la política agrega límites, excepciones y autoridad humana.

applications = pd.read_csv(root / 'data' / 'applications.csv')
policy = apply_review_policy(applications)
policy.to_csv(root / 'submission' / 'review_policy.csv', index=False)

contract = policy_contract()
(root / 'submission' / 'policy_contract.json').write_text(
    json.dumps(contract, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
contract


{'policy_version': 'P303-v1',
 'decision': 'revisar solicitudes de crédito',
 'cadence': 'cada solicitud recibida',
 'response_target': 'recomendación durante la revisión de la solicitud',
 'actions': ['recomendar_aprobación',
  'recomendar_rechazo',
  'escalar_a_revisión_humana'],
 'safeguards': ['no se automatiza la decisión crediticia',
  'la documentación incompleta siempre escala',
  'los montos superiores al límite delegable requieren supervisor'],
 'review_trigger': 'resultado, excepción o cambio de política',
 'outcomes_to_monitor': ['incumplimiento', 'tiempo de decisión']}

In [3]:
policy[[
    'application_id',
    'default_probability',
    'requested_amount',
    'action',
    'reason',
    'human_authority',
]]


,application_id,default_probability,requested_amount,action,reason,human_authority
0,A01,0.04,4000,recomendar_aprobación,riesgo bajo dentro del límite delegable,analista_de_crédito
1,A02,0.11,8500,escalar_a_revisión_humana,riesgo intermedio: se requiere juicio documentado,analista_de_crédito
2,A03,0.19,2500,escalar_a_revisión_humana,riesgo intermedio: se requiere juicio documentado,analista_de_crédito
3,A04,0.28,15000,escalar_a_revisión_humana,monto superior al límite delegable,supervisor_de_crédito
4,A05,0.07,12000,escalar_a_revisión_humana,documentación incompleta,gestor_de_documentación
5,A06,0.33,3000,escalar_a_revisión_humana,documentación incompleta,gestor_de_documentación
